# Thursday · F1 forecasts, AI check and temporal validation

IIT414W · 24 September 2026 · 12:30–15:00

**Do:** close Italy–Spain; write Form/Content/Utility criteria before auditing a sample AI answer; forecast Azerbaijan and the Bahrain GP in Malaysia separately; then finish the leakage and validation work below. **Why:** evidence and its time of availability determine whether a forecast or test is credible.

**Hand in at 15:00:** the completed `IIT414W_W03_Thu_F1_AI_Worksheet_Student_v2.md` on paper. A digital accommodation can be shown to the lecturer then. Keep a copy of the new forecasts. Save this notebook for study; it is not a second submission. No grade.

**Route:** 12:35 old records/results → 12:50 AI check and two forecasts → 13:25 leakage → 13:35 break → 13:45 audit → 14:00 validation → 14:45 exit and hand-in.

Keep the 2022 check and 2023–2024 final test in their original Lab 1 roles. Work through one written step at a time.


## F1 closure and two new forecasts

Use the separate class record for all written answers. **First** copy your 3 and 10 September cards or mark missing values “not recorded”. **Next** calculate the observed Italy–Spain result using the local data below. **Then** write your three AI criteria before looking at the deliberately flawed example. Complete both separate GP rows and retain a copy. Return to this notebook for the validation questions.


In [1]:
from pathlib import Path
import pandas as pd

def locate_prediction_data():
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        candidate = base / "data" / "samples" / "w03_prediction_v1" / "recent_race_points.csv"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Keep data/samples/w03_prediction_v1 in the extracted student pack.")

recent = pd.read_csv(locate_prediction_data())
by_race = recent.groupby(["event", "team"], sort=False)["race_points"].sum().unstack()
by_race["gap_F_minus_M"] = by_race["Ferrari"] - by_race["McLaren"]
display(by_race.loc[["Netherlands", "Italy", "Spain"]])
print("Italy + Spain gap:", int(by_race.loc[["Italy", "Spain"], "gap_F_minus_M"].sum()))


team,McLaren,Ferrari,gap_F_minus_M
event,,,
Netherlands,33,22,-11
Italy,22,8,-14
Spain,19,12,-7


Italy + Spain gap: -21


Write in the forecast sheet: (a) whether each recorded winner was right, (b) absolute margin error only when a numerical margin was actually recorded, and (c) two new forecasts, one for Azerbaijan and one for the Bahrain GP at Sepang. Record each source and its availability date. Freeze both forecasts before discussing later updates.

---

## Lab 1: availability, leakage and validation


## 1 · Prediction moment and data boundary

Lab 1 predicted final top-10 classification after qualifying and before the race. Write one feature known at that moment and one outcome that did not yet exist. Your submitted Lab 1 remains unchanged.

**Known before the race:**

**Known only afterward:**

**One decision I froze before opening test:**

In [2]:
from pathlib import Path
import pandas as pd

def locate_week3_data():
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        candidate = base / "data" / "samples" / "w03_v1"
        if (candidate / "train_results_v1.csv").exists():
            return candidate
        candidate = base / "w03_v1"
        if (candidate / "train_results_v1.csv").exists():
            return candidate
    raise FileNotFoundError("Keep w03_v1/ beside this notebook or inside data/samples/.")

DATA_DIR = locate_week3_data()
print("Local data:", DATA_DIR)

Local data: C:\Users\bvial\Desktop\AI-Workshop\W03_Thu_StudentPack\data\samples\w03_v1


In [3]:
results = pd.read_csv(DATA_DIR / "train_results_v1.csv")
qualifying = pd.read_csv(DATA_DIR / "train_qualifying_v1.csv")
KEY = ["season", "round", "driver_id"]
print("Result rows:", len(results), "Qualifying rows:", len(qualifying))
print("Seasons:", sorted(results["season"].unique().tolist()))
print("Duplicate result keys:", results.duplicated(KEY).sum())
print("Duplicate qualifying keys:", qualifying.duplicated(KEY).sum())
print("Result rows:", len(results))
display(results.tail())
print("Qualifying rows:", len(qualifying))
display(qualifying.tail())

Result rows: 1200 Qualifying rows: 1197
Seasons: [2019, 2020, 2021]
Duplicate result keys: 0
Duplicate qualifying keys: 0
Result rows: 1200


,season,round,race_name,circuit_id,race_date,driver_id,driver_name,constructor_id,grid,position,position_text,status,points,laps
1195,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,latifi,Nicholas Latifi,williams,16,16,R,Accident,0.0,50
1196,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,giovinazzi,Antonio Giovinazzi,alfa,14,17,R,Gearbox,0.0,33
1197,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,russell,George Russell,williams,17,18,R,Gearbox,0.0,26
1198,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,raikkonen,Kimi Räikkönen,alfa,18,19,R,Brakes,0.0,25
1199,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,mazepin,Nikita Mazepin,haas,20,20,W,Illness,0.0,0


Qualifying rows: 1197


,season,round,race_name,circuit_id,race_date,driver_id,driver_name,constructor_id,qualifying_position,q1,q2,q3
1192,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,latifi,Nicholas Latifi,williams,16,1:24.338,NaN,NaN
1193,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,russell,George Russell,williams,17,1:24.423,NaN,NaN
1194,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,raikkonen,Kimi Räikkönen,alfa,18,1:24.779,NaN,NaN
1195,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,mick_schumacher,Mick Schumacher,haas,19,1:24.906,NaN,NaN
1196,2021,22,Abu Dhabi Grand Prix,yas_marina,2021-12-12,mazepin,Nikita Mazepin,haas,20,1:25.685,NaN,NaN


A result row is one driver in one race. The result table contains post-race fields because it is a historical source. Their presence in a file does not make them valid pre-race predictors.

In [5]:
# Combine race results with qualifying data using the shared identifying columns.
# A left join keeps every row from the results table, even if qualifying data is missing.
# validate="one_to_one" checks that each result key matches at most one qualifying row.
# indicator=True adds "_merge" to show whether each row matched successfully.
merged = results.merge(
    qualifying[KEY + ["qualifying_position"]], on=KEY,
    how="left", validate="one_to_one", indicator=True
)

# Count how many rows matched and how many lacked a qualifying record.
print(merged["_merge"].value_counts().to_string())

# Count missing qualifying positions after the merge.
# Missing values represent unavailable qualifying information, not a known zero.
print("Missing qualifying position:", merged["qualifying_position"].isna().sum())

# Summarize result rows and missing qualifying positions for each season.
# This helps identify whether missingness is concentrated in particular seasons.
display(merged.groupby("season", as_index=False).agg(
    result_rows=("driver_id", "size"),
    missing_qualifying=("qualifying_position", lambda s: int(s.isna().sum()))
))

_merge
both          1197
left_only        3
right_only       0
Missing qualifying position: 3


,season,result_rows,missing_qualifying
0,2019,420,2
1,2020,340,0
2,2021,440,1


**Check:** What would change if the join were `inner`? Would missing qualifying data become a known zero, or disappear from the evaluated population? Write a one-sentence answer here.

## 2 · Feature availability audit

For each candidate feature, write the earliest time it could be known and whether it is valid for this prediction. Explain one answer to a partner before seeing their answer.

| Candidate | Earliest availability | Use before the race? Why? |
|---|---|---|
| `qualifying_position` | | |
| `position` from this race | | |
| `status` from this race | | |
| Driver mean finish computed using all 2019–2024 rows | | |
| Constructor points accumulated before this race, with a documented timestamp | | |

Record one correction to your initial judgement:

## 3 · Leakage hunt

A fictional analyst merges 2019–2024, builds `target_top10` from final position, creates `driver_mean_finish` using every season, fits a missing-value imputer on all rows, shuffles rows into train/test, then chooses the best rule by repeatedly checking test accuracy.

1. Identify two distinct defects. For each, name the information that crosses the boundary.
2. Repair the order of decisions. What must be frozen before opening 2023–2024?
3. The analyst drops records without qualifying data. Which population would the reported score then describe?

| Step | Boundary crossed | Repair |
|---|---|---|
| | | |
| | | |

**Population after dropping missing qualifying rows:**

**Decision to freeze before final test:**

## 4 · Validation cuts inside train

The lab uses 2019–2021 for development, 2022 for a separate check and 2023–2024 for final test. The table below describes only practice folds within the 2019–2021 development period. It does not turn 2022 or 2023–2024 into tuning folds.

In [5]:
folds = pd.DataFrame([
    {"design": "expanding", "fold": 1, "train_years": "2019", "validation_years": "2020"},
    {"design": "expanding", "fold": 2, "train_years": "2019–2020", "validation_years": "2021"},
    {"design": "sliding, fixed one-year history", "fold": 1, "train_years": "2019", "validation_years": "2020"},
    {"design": "sliding, fixed one-year history", "fold": 2, "train_years": "2020", "validation_years": "2021"},
])
display(folds)
print("Train years available:", sorted(results["season"].unique().tolist()))

,design,fold,train_years,validation_years
0,expanding,1,2019,2020
1,expanding,2,2019–2020,2021
2,"sliding, fixed one-year history",1,2019,2020
3,"sliding, fixed one-year history",2,2020,2021


Train years available: [2019, 2020, 2021]


**Your decision:** What changes between expanding and sliding? Which design would you inspect first if an older season may be less relevant, and why? With only three years of development data, what can these two folds not establish?

## 5 · Gap and purging

Consider a different prediction task where each training label summarizes events over several days. A row dated 29 December may use information through 3 January. A validation period starts 1 January. The row's timestamp is before validation, but its information interval overlaps validation.

| Row | Timestamp | Information interval | Validation starts |
|---|---|---|---|
| A | 29 Dec | 29 Dec–3 Jan | 1 Jan |
| B | 20 Dec | 20–23 Dec | 1 Jan |

Which row needs exclusion or a sufficient gap? Explain the overlap. Would you automatically add the same gap to every F1 dataset? What evidence about the feature/label construction would you need?

In [6]:
intervals = pd.DataFrame([
    {"row": "A", "start_day": -3, "end_day": 2},
    {"row": "B", "start_day": -12, "end_day": -9},
])
validation_start = 0
intervals["overlaps_validation"] = intervals["end_day"] >= validation_start
display(intervals)

,row,start_day,end_day,overlaps_validation
0,A,-3,2,True
1,B,-12,-9,False


## Exit record

**One leakage defect I can now explain:**

**One rule that protects the final test:**

**One question for Friday:**

Save this notebook. No separate graded submission is created by today's activity.